# Sliding Puzzle Verification\nVerify goal state `[0,1,2,...,N-1]`, solvability check, and all difficulty configs.

In [1]:
import sys
sys.path.insert(0, "../src/sliding_puzzle")
from enviroment import SlidingPuzzle, CONFIGS

## Test all difficulty configs

In [2]:
for name, state in CONFIGS.items():
    print(f"=== {name} ===")
    try:
        puzzle = SlidingPuzzle(initial_state=state)
        n = puzzle.grid_size
        expected_goal = list(range(puzzle.size))
        assert puzzle.goal_state == expected_goal, f"Goal mismatch: {puzzle.goal_state} != {expected_goal}"
        print(f"  Goal: {puzzle.goal_state} ✓")
        print(f"  Initial: {puzzle.get_state()}")
        print(f"  Grid:\n{puzzle.visualize()}")
        print(f"  Solved: {puzzle.is_solved()}")
        print(f"  Progress: {puzzle.compute_progress():.1%}")
        print()
    except ValueError as e:
        print(f"  ✗ FAILED: {e}\n")

=== 2x2 ===
  Goal: [0, 1, 2, 3] ✓
  Initial: [2, 1, 3, 0]
  Grid:
[2, 1]
[3, 0]
  Solved: False
  Progress: 25.0%

=== 3x3 (easiest) ===
  Goal: [0, 1, 2, 3, 4, 5, 6, 7, 8] ✓
  Initial: [1, 2, 5, 6, 3, 4, 7, 8, 0]
  Grid:
[1, 2, 5]
[6, 3, 4]
[7, 8, 0]
  Solved: False
  Progress: 0.0%

=== 3x3 (hardest) ===
  Goal: [0, 1, 2, 3, 4, 5, 6, 7, 8] ✓
  Initial: [8, 6, 7, 2, 5, 4, 3, 1, 0]
  Grid:
[8, 6, 7]
[2, 5, 4]
[3, 1, 0]
  Solved: False
  Progress: 0.0%

=== 4x4 (easiest) ===
  Goal: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] ✓
  Initial: [1, 2, 3, 7, 8, 4, 5, 6, 9, 10, 11, 15, 0, 12, 13, 14]
  Grid:
[1, 2, 3, 7]
[8, 4, 5, 6]
[9, 10, 11, 15]
[0, 12, 13, 14]
  Solved: False
  Progress: 0.0%

=== 4x4 (hardest) ===
  Goal: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] ✓
  Initial: [15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
  Grid:
[15, 14, 13, 12]
[11, 10, 9, 8]
[7, 6, 5, 4]
[3, 2, 1, 0]
  Solved: False
  Progress: 0.0%



## Test move + solve check

In [3]:
# One-move solve: 3x3
puzzle = SlidingPuzzle(initial_state=[1, 0, 2, 3, 4, 5, 6, 7, 8])
print("Before:", puzzle.get_state())
print(puzzle.visualize())
puzzle.move_tile(1, "right")
print("\nAfter move_tile(1, 'right'):", puzzle.get_state())
print(puzzle.visualize())
print(f"Solved: {puzzle.is_solved()} ✓" if puzzle.is_solved() else "NOT solved ✗")

Before: [1, 0, 2, 3, 4, 5, 6, 7, 8]
[1, 0, 2]
[3, 4, 5]
[6, 7, 8]

After move_tile(1, 'right'): [0, 1, 2, 3, 4, 5, 6, 7, 8]
[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
Solved: True ✓


## Test unsolvable rejection

In [4]:
# These should all raise ValueError
unsolvable = {
    "3x3 odd inversions": [1, 2, 3, 4, 5, 6, 8, 7, 0],
    "2x2 unsolvable": [1, 2, 3, 0],  # even inversions but wrong blank row parity for even-width
}

for name, state in unsolvable.items():
    try:
        SlidingPuzzle(initial_state=state)
        print(f"  ✗ {name}: should have been rejected!")
    except ValueError:
        print(f"  ✓ {name}: correctly rejected")

  ✓ 3x3 odd inversions: correctly rejected
  ✓ 2x2 unsolvable: correctly rejected


## Test generic prompts

In [5]:
sys.path.insert(0, "../src/sliding_puzzle")
from prompts import get_system_prompt, build_user_prompt

for name, state in [("3x3", [8, 6, 7, 2, 5, 4, 3, 1, 0]), ("4x4", [15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0])]:
    puzzle = SlidingPuzzle(initial_state=state)
    print(f"=== {name} System Prompt (first 500 chars) ===")
    print(get_system_prompt(puzzle)[:500])
    print(f"\n=== {name} User Prompt ===")
    print(build_user_prompt(puzzle.get_state(), None, puzzle, 1))
    print("-" * 60)

=== 3x3 System Prompt (first 500 chars) ===

You are a micro-agent solving a 8-puzzle (sliding puzzle) on a 3x3 grid.

Goal State: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Visualized:
 0  1  2
 3  4  5
 6  7  8

The state is a flat list in row-major order:
  Index 0 = top-left (tile 0)
  Index 1 = top-center (tile 1)
  Index 2 = top-right (tile 2)
  Index 3 = mid-left (tile 3)
  Index 4 = mid-center (tile 4)
  Index 5 = mid-right (tile 5)
  Index 6 = bot-left (tile 6)
  Index 7 = bot-center (tile 7)
  Index 8 = bot-right (tile 8)

Directions (the dire

=== 3x3 User Prompt ===

Step: 1
Current State: [8, 6, 7, 2, 5, 4, 3, 1, 0]
Previous Move: None (first move)

Grid:
 8  6  7
 2  5  4
 3  1  0

Task:
1. Scan tiles 1->8. Find the first tile not at its goal index. That is your Target.
2. Apply Move Logic (Rule 2) to move Target closer to its goal.
3. Verify your move is legal and next_state is correct.

Provide Target, Reasoning, move, and next_state:

-------------------------------------------------